In [ ]:
pip install -U transformers datasets accelerate torch scikit-learn

In [ ]:
conda install pytorch>=2.6 torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia

In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-mix"
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 4
LR = 2e-5
OUTPUT_DIR = "./deberta-translation-pedagogy"

dataset = load_dataset(
    "csv",
    data_files="intent_data.csv",
)

dataset = dataset["train"].train_test_split(
    test_size=0.1,
    seed=42,
)

def normalize_labels(batch):
    batch["classes"] = [c.strip().upper() for c in batch["classes"]]
    return batch

dataset = dataset.map(normalize_labels, batched=True)

label2id = {
    "TRANSLATION": 0,
    "PEDAGOGY_CATEGORY": 1,
}
id2label = {v: k for k, v in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast = True)

def preprocess(batch):
    return tokenizer(
        batch["utterance"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

dataset = dataset.map(preprocess, batched=True)

def encode_labels(batch):
    batch["labels"] = [label2id[label] for label in batch["classes"]]
    return batch

dataset = dataset.map(encode_labels, batched=True)

dataset = dataset.remove_columns(["utterance", "classes"])
dataset.set_format("torch")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    label2id=label2id,
    id2label=id2label,
    use_safetensors = True
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    logging_steps=50,
    logging_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


In [ ]:
from transformers import pipeline

model_path = "./deberta-translation-pedagogy"
classifier = pipeline(
    "text-classification",
    model = model_path,
    tokenizer = model_path,
    return_all_scores = True
)

text = ["ترجم هذه الكلمة الى لغة الاشارة",
        "كيف احكي اكلت هريسة بلغة الاشارة",
        "أعطيني إشارات وسائل الإعلام",
        "بدي اتعلم كل الاشارات المرتبطة بالسوق",
        "ابغى اشارات المرمطة",
        "اريد منك ان تعلمني "]

results = classifier(text)

for i, j in results:
    print(i, "and ", j)

Device set to use cuda:0


{'label': 'TRANSLATION', 'score': 0.5614653825759888} and  {'label': 'PEDAGOGY_CATEGORY', 'score': 0.4385346472263336}
{'label': 'TRANSLATION', 'score': 0.5763416886329651} and  {'label': 'PEDAGOGY_CATEGORY', 'score': 0.4236583709716797}
{'label': 'TRANSLATION', 'score': 0.42183569073677063} and  {'label': 'PEDAGOGY_CATEGORY', 'score': 0.5781643390655518}
{'label': 'TRANSLATION', 'score': 0.5224761366844177} and  {'label': 'PEDAGOGY_CATEGORY', 'score': 0.47752389311790466}
{'label': 'TRANSLATION', 'score': 0.4767117202281952} and  {'label': 'PEDAGOGY_CATEGORY', 'score': 0.5232882499694824}
